In [4]:
import load_env
import os
dataset = os.environ.get("WORKSPACE_CDR")


In [5]:
import pandas_gbq
import pandas as pd

query = f"""
WITH z3a_mapping AS (
    SELECT
        icd.concept_code,
        std.concept_id AS standard_concept_id
    FROM `{dataset}.concept` icd
    JOIN `{dataset}.concept_relationship` cr
        ON icd.concept_id = cr.concept_id_1
    JOIN `{dataset}.concept` std
        ON cr.concept_id_2 = std.concept_id
    WHERE icd.concept_code LIKE 'Z3A.%'
      AND cr.relationship_id = 'Maps to'
)

SELECT
    co.person_id,
    co.condition_start_date,
    z.concept_code
FROM `{dataset}.condition_occurrence` co
JOIN z3a_mapping z
    ON co.condition_concept_id = z.standard_concept_id
"""

df1 = pandas_gbq.read_gbq(query)
df1 = df1.sort_values(["person_id", "condition_start_date", "concept_code"])
df1

Downloading: 100%|██████████|


,person_id,condition_start_date,concept_code
102382,1000131,2021-07-27,Z3A.00
191949,1000131,2021-07-27,Z3A.2
736579,1000131,2021-07-27,Z3A.49
103654,1000131,2021-08-02,Z3A.00
193221,1000131,2021-08-02,Z3A.2
...,...,...,...
589741,9999860,2017-07-07,Z3A.4
605883,9999860,2017-07-07,Z3A.4
681434,9999860,2017-07-07,Z3A.4
682789,9999860,2017-07-07,Z3A.4


In [6]:
def z3a_to_weeks(code):
    if code in {"Z3A.00", "Z3A.0", "Z3A.1", "Z3A.2","Z3A.3","Z3A.4","Z3A.49"}:
        return None
    if code == "Z3A.01":
        return 7
    return int(code.split(".")[1])

df1["weeks"] = df1["concept_code"].apply(z3a_to_weeks)

df1 = df1[df1["weeks"].notna()]

df1 = df1.drop_duplicates(
    subset=["person_id", "condition_start_date", "concept_code", "weeks"]
)

df1


,person_id,condition_start_date,concept_code,weeks
521849,1000131,2021-09-08,Z3A.39,39.0
520590,1000131,2021-09-22,Z3A.39,39.0
167927,1000195,2020-12-28,Z3A.13,13.0
167753,1000195,2021-01-28,Z3A.13,13.0
177288,1000195,2021-03-04,Z3A.18,18.0
...,...,...,...,...
476611,9999860,2017-06-16,Z3A.36,36.0
8893,9999860,2017-06-23,Z3A.17,17.0
491151,9999860,2017-06-27,Z3A.37,37.0
504676,9999860,2017-06-30,Z3A.38,38.0


In [7]:
#estimate LMP using formula: start-weeks of gestation
df1["estimated_lmp"] = (
    pd.to_datetime(df1["condition_start_date"])
    - pd.to_timedelta(df1["weeks"] * 7, unit="D")
)

#sort by person to see range in LMP
df1 = df1.sort_values(["person_id", "condition_start_date", "concept_code"])

df1

,person_id,condition_start_date,concept_code,weeks,estimated_lmp
521849,1000131,2021-09-08,Z3A.39,39.0,2020-12-09
520590,1000131,2021-09-22,Z3A.39,39.0,2020-12-23
167927,1000195,2020-12-28,Z3A.13,13.0,2020-09-28
167753,1000195,2021-01-28,Z3A.13,13.0,2020-10-29
177288,1000195,2021-03-04,Z3A.18,18.0,2020-10-29
...,...,...,...,...,...
476611,9999860,2017-06-16,Z3A.36,36.0,2016-10-07
8893,9999860,2017-06-23,Z3A.17,17.0,2017-02-24
491151,9999860,2017-06-27,Z3A.37,37.0,2016-10-11
504676,9999860,2017-06-30,Z3A.38,38.0,2016-10-07


In [8]:
from tqdm import tqdm
tqdm.pandas()

def cluster_lmps(df):
    df = df.sort_values("estimated_lmp").copy()

    cluster = 0
    cluster_start = df.iloc[0]["estimated_lmp"] #initialze as first (earlistest) lmp
    labels = []

    for lmp in df["estimated_lmp"]:
        if (lmp - cluster_start).days > 42: #6 weeks =42 days
            cluster += 1
            cluster_start = lmp #restart cluster

        labels.append(cluster)

    df["cluster"] = labels
    return df

#get clusters and get median of LMPs
clustered = (df1.groupby("person_id", group_keys=False)
             .progress_apply(cluster_lmps)
             .reset_index(drop=True))

/opt/conda/envs/jupyter/lib/python3.10/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)
100%|██████████| 21478/21478 [00:35<00:00, 604.22it/s]


In [9]:
#get median for each cluster

def median(x):
    # vals = x.sort_values().astype("int64")
    # return pd.to_datetime(vals.iloc[(len(vals)-1)//2])
    return pd.to_datetime(x).median()

lmp_summary = (clustered
               .groupby(["person_id", "cluster"], as_index=False)
               .agg(median_lmp=("estimated_lmp", median),
                    n_records=("estimated_lmp", "size"), #amount of codes
                    min_lmp=("estimated_lmp", "min"),
                    max_lmp=("estimated_lmp", "max")))



In [10]:
# calculate reliability score

def lmp_score(lmps):
    lmps = pd.to_datetime(lmps).sort_values().reset_index(drop=True)
    n = len(lmps)

    # Median LMP
    med = median(lmps)

    #S1= number of obseravtions
    S1 = min(n/10, 1)

    #S2= % of observations that are within 7 days of the median
    distances = (lmps - med).dt.days.abs()
    S2 = (distances <= 7).mean()

    #S3= difference between mode and mean
    mode_val = lmps.mode().iloc[0]
    d = abs((med - mode_val).days)
    S3 = max(0, 1 - d/7)

    #S4= % of observatios thats the mode
    p = (lmps == mode_val).mean()
    S4 = p

    score = 100 * 0.25 * (S1+ S2 + S3 + S4)

    return round(score, 1)

In [11]:
#clusters within 8 weeks are merged\

def merge_clusters(df):
    df = df.sort_values("median_lmp").reset_index(drop=True)
    merged = []
    i = 0

    while i < len(df):
        current = df.iloc[i].copy()
        if i < len(df)-1: # check if theres another cluster after this one
            nxt = df.iloc[i+1]
            if (nxt["median_lmp"] - current["median_lmp"]).days <= 56:
                all_rows = pd.concat([clustered[(clustered.person_id == current.person_id) &
                                       (clustered.cluster == current.cluster)],
                                       clustered[(clustered.person_id == nxt.person_id) &
                                       (clustered.cluster == nxt.cluster)]]) #retrieve lmp estimates from both clusters
                all_dates = all_rows["estimated_lmp"]
                last_record = all_rows.sort_values("condition_start_date").iloc[-1]
                merged.append({
                    "person_id": current.person_id,
                    "episode_id": len(merged),
                    "median_lmp": median(all_dates),
                    "score": lmp_score(all_dates),
                    # "n_clusters": 2,
                    "n_records": len(all_dates),
                    "min_lmp": all_dates.min(),
                    "max_lmp": all_dates.max(),
                    "last_zcode": last_record["concept_code"],
                    "last_zcode_date": last_record["condition_start_date"]})
                i += 2 #skip next cluster (merged)
                continue
                
        # keep cluster as its own episode
        all_rows = clustered[(clustered.person_id == current.person_id) &
                    (clustered.cluster == current.cluster)]
        all_dates = all_rows["estimated_lmp"]
        last_record = all_rows.sort_values("condition_start_date").iloc[-1]
        
        merged.append({
            "person_id": current.person_id,
            "episode_id": len(merged),
            "median_lmp": current.median_lmp,
            "score": lmp_score(all_dates),
            # "n_clusters": 1,
            "n_records": current.n_records,
            "min_lmp": current.min_lmp,
            "max_lmp": current.max_lmp,
            "last_zcode": last_record["concept_code"],
            "last_zcode_date": last_record["condition_start_date"]})
        i += 1
#im going to LOSE my midn
    return pd.DataFrame(merged)

#apply for each person
final_episodes = (lmp_summary.groupby("person_id", group_keys=False)
                  .progress_apply(merge_clusters)
                  .reset_index(drop=True))

/opt/conda/envs/jupyter/lib/python3.10/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)
100%|██████████| 21478/21478 [03:33<00:00, 100.82it/s]


In [12]:
final_episodes

,person_id,episode_id,median_lmp,score,n_records,min_lmp,max_lmp,last_zcode,last_zcode_date
0,1000131,0,2020-12-16 00:00:00,42.5,2,2020-12-09,2020-12-23,Z3A.39,2021-09-22
1,1000195,0,2020-10-29 00:00:00,90.6,16,2020-09-28,2020-11-03,Z3A.40,2021-08-10
2,1000210,0,2015-03-19 00:00:00,78.8,4,2015-03-12,2015-03-19,Z3A.39,2015-12-17
3,1000210,1,2016-09-08 00:00:00,87.5,16,2016-09-01,2016-10-13,Z3A.42,2017-06-27
4,1000210,2,2022-10-26 00:00:00,78.8,4,2022-10-25,2022-10-26,Z3A.37,2023-07-12
...,...,...,...,...,...,...,...,...,...
33515,9998770,0,2016-01-20 12:00:00,49.6,2,2016-01-15,2016-01-26,Z3A.39,2016-10-25
33516,9998770,1,2021-08-14 12:00:00,60.4,2,2021-08-12,2021-08-17,Z3A.38,2022-05-05
33517,9999029,0,2020-02-18 12:00:00,67.5,2,2020-02-18,2020-02-19,Z3A.16,2020-06-10
33518,9999860,0,2016-10-10 00:00:00,71.0,15,2016-10-06,2016-10-17,Z3A.39,2017-07-07
